# Notebook 01 - Data Collection

## Objective

Collect, inspect, and validate the NBA datasets used to build player archetype profiles.

The project uses NBA base, advanced, tracking, shot-location, hustle, and matchup data to capture playing style behavior.

### Goals

1. Document the data sources.
2. Define the eligible player population.
3. Validate dataset structure.
4. Evaluate missing data and player coverage.
5. Identify data limitations before feature engineering.

No modeling or final feature constrution is performed here.

### Data Collection

NBA API requests are implemented in 'src/nba_archetypes/data_collection.py'

Raw responses are cached in 'data/raw/2025_26' as Parquet files. (Year changes depending on SEASON variable set in 'src/nba_archetypes/config.py')

This notebook loads the cached datasets rather than repeatedly requesting data from the NBA API.

In [1]:
import pandas as pd
import numpy as np

from nba_archetypes.config import RAW_DATA_DIR, INTERIM_DATA_DIR, METADATA_DIR, MIN_MINS

import nba_archetypes.data_collection as dc

from nba_archetypes.data_validation import check_unique_ids, player_coverage_report


### 1. Data Sources and Collection

Player statistics were collected using the Python 'nba_api' package.

The collection pipeline retrieves several categories of data:
- Traditional and advanced player statistics
- Usage data and scoring splits
- Offensive tracking data
- Shot-location data
- Hustle and defensive tracking data
- Defensive player matchups

API requests and caching are managed through a separate Python module.

Previously collected CSV datasets were migrated to Parquet where necessary (as apart of a project restructure). Newly collected datasets are cached directly as Parquet.

This allows the analysis to run without repeatedly making external API requests.

In [6]:
dc.get_player_stats()

datasets = {}
for file in RAW_DATA_DIR.glob("*.parquet"):
    dataset_name = file.stem
    datasets[dataset_name] = pd.read_parquet(file)

Change force to True if datasets don't already exist.


### 2. Dataset Inventory

Before cleaning or merging the datasets, let's examine the available source tables.

The inventory records the number of rows and columns in each dataset and helps identify missing or imcomplete sources.

In [7]:
inventory = []

for dataset_name, df in datasets.items():
    inventory.append({
        "Dataset": dataset_name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": df.isna().sum().sum()
    })

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

,Dataset,Rows,Columns,Missing Values
0,advanced_stats,582,79,0
1,base_stats,582,69,0
2,catchshoot_stats,582,16,140
3,drives_stats,582,25,0
4,efficiency_stats,582,22,0
5,hustle_stats,581,28,0
6,matchup_stats,147805,27,0
7,painttouch_stats,582,26,0
8,passing_stats,582,18,0
9,possessions_stats,582,21,0


### 3. Player Eligibility

This project includes players with at least 250 regular season minutes.

This threshold is intended to reduce the influence of very small sample size players while retaining a resonably broad pool of players to work with.

The threshold will be evaluated for sensitivity later in the modeling process.

#### Core Player Data

Our base and advanced stats tables will be used to establish the core player population and provide context such as minutes, games played, and possessions.

These datasets will also provide the identifiers needed to combine our other data sources.

In [4]:
eligible_players = datasets["base_stats"].loc[datasets["base_stats"]["MIN"] >= MIN_MINS].copy()

print("Total Players:", len(datasets["base_stats"]))
print("Eligible Players:", len(eligible_players))

display(eligible_players.head())

Total Players: 582
Eligible Players: 450


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,FP_HIGH_SCORE_RANK,TEAM_COUNT
1,1631260,AJ Green,AJ,1610612749,MIL,26.0,78,31,47,0.397,...,540,229,126,496,175,284,38,139,167,1
2,1642358,AJ Johnson,AJ,1610612742,DAL,21.0,48,9,39,0.188,...,158,384,400,394,414,284,38,413,412,2
3,203932,Aaron Gordon,Aaron,1610612743,DEN,30.0,36,27,9,0.750,...,242,143,197,48,247,114,38,241,250,1
4,1628988,Aaron Holiday,Aaron,1610612745,HOU,29.0,57,38,19,0.667,...,274,323,321,99,354,284,38,348,352,1
5,1630174,Aaron Nesmith,Aaron,1610612754,IND,26.0,45,10,35,0.222,...,409,156,184,520,240,284,38,221,243,1


### 4. Player Data Coverage

Not every NBA dataset is guaranteed to include every eligible player.

I measured the percentage of eligible players represented in each data source before deciding whether those variables should be included in our final model.

In [8]:
player_coverage_report(datasets, eligible_players)

,dataset,coverage_percentage
0,advanced_stats,100.0
1,base_stats,100.0
2,catchshoot_stats,100.0
3,drives_stats,100.0
4,efficiency_stats,100.0
5,hustle_stats,100.0
6,matchup_stats,100.0
7,painttouch_stats,100.0
8,passing_stats,100.0
9,possessions_stats,100.0


In [10]:
missing_pct = (datasets["pullupshot_stats"].isna().mean() * 100).sort_values(ascending=False)

display(missing_pct.head())

PULL_UP_FG3_PCT    17.182131
PULL_UP_FG_PCT      7.903780
PULL_UP_EFG_PCT     7.903780
PULL_UP_FG3A        6.013746
PULL_UP_FG3M        6.013746
dtype: float64